In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/happiness/landing/archivos/ILOSTAT/EIP_NEET_SEX_RT_A-20261121T1414.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
)
;

In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/happiness/landing/archivos/ILOSTAT/LUU_2LU2_SEX_RT_A-20261121T1414.csv',
  format => 'csv'
) LIMIT 5;


In [0]:
%sql
SELECT * FROM read_files(
  '/Volumes/happiness/landing/archivos/ILOSTAT/EIP_NEET_SEX_RT_A-20261121T1414.csv',
  format => 'csv'
) LIMIT 5;

In [0]:
%sql
-- =========================================================
-- happiness.bronze.ilostat_luu
-- Subutilización de la fuerza laboral (LU2) — LUU_2LU2_SEX_RT_A
-- Full Load: CREATE OR REPLACE reemplaza la tabla entera, no duplica.
-- inferColumnTypes => false desde el vamos (D15) -> nace 100% STRING (D10).
-- Wildcard en el nombre de archivo porque la fecha del nombre cambia en cada
-- descarga manual (ej. LUU_2LU2_SEX_RT_A-20261121T1414.csv).
-- =========================================================
CREATE OR REPLACE TABLE happiness.bronze.ilostat_luu
COMMENT 'Bronze ILOSTAT - subutilización de la fuerza laboral (indicador LU2, LUU_2LU2_SEX_RT_A). Crudo tal como lo sirve rplumber.ilo.org, sin cast ni filtro de filas. Lista corta ILOSTAT, ver D6.'
AS
SELECT
  *,
  current_timestamp()                        AS _ingest_ts,
  _metadata.file_name                       AS _source
FROM read_files(
  '/Volumes/happiness/landing/archivos/ILOSTAT/LUU_2LU2_SEX_RT_A*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
);

ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN ref_area    COMMENT 'Código de área/país tal como lo da ILOSTAT (no siempre ISO3 limpio; se concilia en Silver)';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN source      COMMENT 'Código de fuente de la observación + referencia de nota, ej. XA:2198';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN indicator   COMMENT 'Código de indicador ILOSTAT, constante = LUU_2LU2_SEX_RT';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN sex         COMMENT 'SEX_T (total), SEX_M (varones) o SEX_F (mujeres)';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN time        COMMENT 'Año de la observación, como STRING (se castea a INT en Silver)';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN obs_value   COMMENT 'Valor del indicador, como STRING (se castea a DECIMAL en Silver)';
ALTER TABLE happiness.bronze.ilostat_luu ALTER COLUMN obs_status  COMMENT 'Flag de calidad del dato (estimado/provisorio); texto "null" literal si no aplica, ver D10';

-- =========================================================
-- happiness.bronze.ilostat_neet
-- Jóvenes que ni estudian ni trabajan — EIP_NEET_SEX_RT_A
-- Mismo patrón; esta fuente SÍ trae note_indicator/note_source.
-- =========================================================
CREATE OR REPLACE TABLE happiness.bronze.ilostat_neet
COMMENT 'Bronze ILOSTAT - jóvenes que ni estudian ni trabajan (NEET, EIP_NEET_SEX_RT_A). Excepción de cobertura D4. Crudo tal como lo sirve rplumber.ilo.org, sin cast ni filtro de filas.'
AS
SELECT
  *,
  current_timestamp()                        AS _ingest_ts,
  _metadata.file_name                           AS _source
FROM read_files(
  '/Volumes/happiness/landing/archivos/ILOSTAT/EIP_NEET_SEX_RT_A*.csv',
  format => 'csv',
  header => true,
  inferColumnTypes => false
);

ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN ref_area       COMMENT 'Código de área/país tal como lo da ILOSTAT (no siempre ISO3 limpio; se concilia en Silver)';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN source         COMMENT 'Código de fuente de la observación + referencia de nota';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN indicator      COMMENT 'Código de indicador ILOSTAT, constante = EIP_NEET_SEX_RT';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN sex            COMMENT 'SEX_T (total), SEX_M (varones) o SEX_F (mujeres)';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN time           COMMENT 'Año de la observación, como STRING (se castea a INT en Silver)';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN obs_value      COMMENT 'Valor del indicador, como STRING (se castea a DECIMAL en Silver)';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN obs_status     COMMENT 'Flag de calidad del dato; texto "null" literal si no aplica, ver D10';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN note_indicator COMMENT 'Código de nota a nivel indicador (metodología), si aplica';
ALTER TABLE happiness.bronze.ilostat_neet ALTER COLUMN note_source    COMMENT 'Código de nota a nivel fuente (ej. cobertura, referencia temporal), si aplica';

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT ref_area) FROM happiness.bronze.ilostat_luu

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT ref_area) FROM happiness.bronze.ilostat_neet